# Career-Match AI | Phase 3

Rank the five benchmark vacancies for each candidate using Sentence Transformers
and cosine similarity. Open this notebook from the repository root and select a
Python environment with the project requirements installed. The first run
downloads `all-MiniLM-L6-v2`.

The input resumes come from `data/raw_resumes/`; each numeric filename supplies
the candidate ID. The vacancies come from `data/5_vacancies.csv`.
The output is `data/ai_rankings.csv`, with one row per candidate–vacancy pair.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sentence_transformers import SentenceTransformer

from src.docx_parser import extract_docx_text

data_dir = Path("data")
vacancies = pd.read_csv(data_dir / "5_vacancies.csv")
resume_paths = sorted(
    (data_dir / "raw_resumes").glob("*.docx"),
    key=lambda path: int(path.stem),
)
candidate_ids = [int(path.stem) for path in resume_paths]
resume_texts = [extract_docx_text(path) for path in resume_paths]
vacancy_texts = [
    f"{job.job_title}\n{job.job_description}"
    for job in vacancies.itertuples(index=False)
]

## Embed the resumes and job descriptions

Split long documents at token boundaries to fit the model's input limit.
Average each document's chunk embeddings, weighted by the number of tokens
in each chunk, then normalize the result so dot products give cosine similarity.

In [2]:
model = SentenceTransformer("all-MiniLM-L6-v2")
tokenizer = model.tokenizer
chunk_size = model.max_seq_length - tokenizer.num_special_tokens_to_add(pair=False)

def embed_documents(texts):
    chunks = []
    chunk_sizes = []
    document_ranges = []

    for text in texts:
        offsets = tokenizer(
            text, add_special_tokens=False, truncation=False,
            return_offsets_mapping=True, verbose=False,
        )["offset_mapping"]
        if not offsets:
            raise ValueError("A resume or vacancy has no text to embed.")

        start_chunk = len(chunks)
        for start in range(0, len(offsets), chunk_size):
            end = min(start + chunk_size, len(offsets))
            chunks.append(text[offsets[start][0]:offsets[end - 1][1]])
            chunk_sizes.append(end - start)
        document_ranges.append((start_chunk, len(chunks)))

    vectors = model.encode(chunks, normalize_embeddings=True)
    documents = np.stack([
        np.average(vectors[start:end], axis=0, weights=chunk_sizes[start:end])
        for start, end in document_ranges
    ])
    return documents / np.linalg.norm(documents, axis=1, keepdims=True)

candidate_vectors = embed_documents(resume_texts)
vacancy_vectors = embed_documents(vacancy_texts)
similarities = candidate_vectors @ vacancy_vectors.T

## Rank the five jobs for each candidate

Rank 1 is the highest cosine similarity. Exact ties follow the order of
`5_vacancies.csv`. Use `vacancy_number` (the 1-based row position) to line up
the results with the annotation file. `vacancy_id` is the ID in the source CSV.
The annotation file covers candidate IDs 1 through 30.

`semantic_score` is raw cosine similarity. Only candidate IDs, vacancy
details, scores, and ranks are exported.

In [3]:
rankings = []
for candidate_id, scores in zip(candidate_ids, similarities, strict=True):
    job_indices = np.argsort(-scores, kind="stable")
    for rank, job_index in enumerate(job_indices, start=1):
        vacancy = vacancies.iloc[job_index]
        rankings.append({
            "candidate_id": candidate_id,
            "vacancy_number": job_index + 1,
            "vacancy_id": vacancy["id"],
            "job_title": vacancy["job_title"],
            "semantic_score": scores[job_index],
            "ai_rank": rank,
        })

results = pd.DataFrame(rankings)
output_path = data_dir / "ai_rankings.csv"
results.to_csv(output_path, index=False, float_format="%.8f")
print(f"Saved {len(results)} rankings for {len(candidate_ids)} candidates to {output_path}.")
display(results.head(len(vacancies)))

Saved 325 rankings for 65 candidates to data/ai_rankings.csv.


,candidate_id,vacancy_number,vacancy_id,job_title,semantic_score,ai_rank
0,1,4,207,Backend Software Developer,0.585045,1
1,1,3,90,Junior Level Software Developer (1-4 years exp...,0.532186,2
2,1,5,499,Software Developer,0.517982,3
3,1,2,37,Remote Software Developer,0.512841,4
4,1,1,8,Software Developer - .Net,0.455461,5
